In [1]:
#!/usr/bin/env python
# coding: utf-8


# TS-SatFire AF Detection v6 -- Clean Data + SE-UNet3D

**Key insight from v1-v5:** The architecture was never the bottleneck.
The F1 ceiling at 0.818 was caused by **18 training fires and 1 val fire
with completely missing AF labels** (band 7 = all NaN). These fires taught
the model to suppress fire predictions, directly capping recall at 0.80.

**v6 changes:**
1. Exclude 18 zero-label train fires (120 clean fires remain)
2. Exclude 1 zero-label val fire (12 clean val fires remain)
3. Skip individual days where band 7 is all-NaN (partial label fires)
4. Same proven SE-UNet3D backbone from v1 (33M params)
5. 80 epochs with OneCycleLR (v1's proven recipe)
6. Post-training threshold sweep

**SOTA claim strategy:** Evaluate on 15 test fires with verified labels.
Report data cleaning as a contribution. Fair apples-to-apples comparison.


In [2]:
# --- Cell 1: Imports ---
import os, gc, sys, time, glob, random, warnings, json, math
from datetime import datetime
from collections import OrderedDict

PIPELINE_START = time.time()

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import f1_score, jaccard_score, precision_score, recall_score
from tqdm.auto import tqdm

try:
    import rasterio
except ImportError:
    os.system("pip install rasterio --quiet")
    import rasterio

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"Device:  {DEVICE}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} -- {p.total_memory/1e9:.1f} GB")
print(f"Setup: {time.time()-PIPELINE_START:.1f}s")


Python:  3.12.12
PyTorch: 2.10.0+cu128
CUDA:    12.8
Device:  cuda
  GPU 0: Tesla T4 -- 15.6 GB
  GPU 1: Tesla T4 -- 15.6 GB
Setup: 7.2s


## Cell 2: Configuration

Same proven recipe as v1. The only difference is data filtering.


In [3]:
class Config:
    DATA_ROOT = "/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire"
    OUTPUT_DIR = "/kaggle/working"
    SAVE_DIR = "/kaggle/working/checkpoints"

    TS_LENGTH = 2
    TRAIN_INTERVAL = 1
    IMAGE_SIZE = 256
    N_CHANNELS = 8
    MEAN = np.array([18.76488, 27.441864, 20.584806, 305.99478,
                     294.31738, 14.625097, 276.4207, 275.16766], dtype=np.float32)
    STD = np.array([15.911591, 14.879259, 10.832616, 21.761852,
                    24.703484, 9.878246, 40.64329, 40.7657], dtype=np.float32)

    SEED = 42
    MAX_EPOCHS = 80
    BATCH_SIZE = 8
    LEARNING_RATE = 5e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 2
    USE_AMP = True

    FOCAL_ALPHA = 0.75
    FOCAL_GAMMA = 2.0
    DICE_WEIGHT = 0.5
    FOCAL_WEIGHT = 0.5
    DS_WEIGHT = 0.3

    ENCODER_CHANNELS = [64, 128, 256, 512]
    DROPOUT = 0.1
    SE_REDUCTION = 8

    MIN_FIRE_PX = 10
    MAX_NEG_RATIO = 2
    PATIENCE = 20

    VAL_IDS = ["20568194", "20701026", "20562846", "20700973", "24462610",
               "24462788", "24462753", "24103571", "21998313", "21751303",
               "22141596", "21999381", "22712904"]

    # Fires with ZERO AF labels (band 7 = all NaN across all days)
    # Identified by our label audit script
    NO_LABEL_IDS = [
        "20777207", "20777386", "21693566", "21751309",
        "21889672", "21889683", "21889697", "21889719",
        "21889734", "21889754", "21997775", "22712973",
        "22713339", "23860939", "23860978", "23861018",
        "23861131", "24332700",
        "22712904",  # val fire with no labels
    ]

cfg = Config()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(cfg.OUTPUT_DIR, "plots"), exist_ok=True)

random.seed(cfg.SEED); np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED); torch.cuda.manual_seed_all(cfg.SEED)

print(f"Model:        SE-UNet3D v6 (clean data)")
print(f"Patch:        {cfg.IMAGE_SIZE}x{cfg.IMAGE_SIZE} center crop")
print(f"Batch:        {cfg.BATCH_SIZE}")
print(f"Epochs:       {cfg.MAX_EPOCHS}")
print(f"LR:           {cfg.LEARNING_RATE}")
print(f"Encoder:      {cfg.ENCODER_CHANNELS}")
print(f"Excluded IDs: {len(cfg.NO_LABEL_IDS)} fires with zero labels")


Model:        SE-UNet3D v6 (clean data)
Patch:        256x256 center crop
Batch:        8
Epochs:       80
LR:           0.0005
Encoder:      [64, 128, 256, 512]
Excluded IDs: 19 fires with zero labels


## Cell 3: Clean Data Loading

**The critical fix:** We filter at TWO levels:
1. **Fire level:** Exclude 19 fires with completely missing labels
2. **Day level:** Skip individual days where band 7 is all-NaN
   (66 partial fires have some good days and some NaN days)

This means the model ONLY trains on verified fire/no-fire labels.


In [4]:
def load_frame(fire_dir, day_path, return_label=False):
    """Load 8-channel frame + optional AF label."""
    with rasterio.open(day_path) as src:
        day_arr = src.read().astype(np.float32)
    day_bands = day_arr[:6]
    label = None
    if return_label and day_arr.shape[0] >= 7:
        b7 = day_arr[6]
        if np.isnan(b7).sum() < b7.size:  # not all NaN
            label = (b7 >= 7).astype(np.float32)

    night_dir = os.path.join(fire_dir, "VIIRS_Night")
    night_path = os.path.join(night_dir,
        os.path.basename(day_path).replace("_VIIRS_Day", "_VIIRS_Night"))
    if os.path.exists(night_path):
        with rasterio.open(night_path) as src:
            na = src.read().astype(np.float32)
        nb = na[:2] if na.shape[0] >= 2 else np.zeros((2, *day_bands.shape[1:]), dtype=np.float32)
    else:
        nb = np.zeros((2, *day_bands.shape[1:]), dtype=np.float32)
    frame = np.concatenate([day_bands, nb], axis=0)
    return (frame, label) if return_label else frame


def check_day_has_label(day_path):
    """Quick check if band 7 has any non-NaN values."""
    with rasterio.open(day_path) as src:
        if src.count < 7:
            return False
        b7 = src.read(7).astype(np.float32)
        return np.isnan(b7).sum() < b7.size


# Build clean train/val splits
all_ids = sorted(os.listdir(cfg.DATA_ROOT))
numeric_ids = [d for d in all_ids if d.isdigit()]

# Exclude fires with no labels
clean_train_ids = [d for d in numeric_ids
                   if d not in cfg.VAL_IDS and d not in cfg.NO_LABEL_IDS]
clean_val_ids = [d for d in numeric_ids
                 if d in cfg.VAL_IDS and d not in cfg.NO_LABEL_IDS]

train_fires = [os.path.join(cfg.DATA_ROOT, d) for d in clean_train_ids]
val_fires = [os.path.join(cfg.DATA_ROOT, d) for d in clean_val_ids]

print(f"Original:  {len(numeric_ids)} fires")
print(f"Excluded:  {len(cfg.NO_LABEL_IDS)} fires (zero labels)")
print(f"Clean train: {len(train_fires)} fires")
print(f"Clean val:   {len(val_fires)} fires")
print(f"Removed from train: {len(numeric_ids) - len(cfg.VAL_IDS) - len(train_fires)} fires")
print(f"Removed from val:   {len(cfg.VAL_IDS) - len(val_fires)} fires")


class AFDatasetClean(Dataset):
    """
    Like v1's dataset but with day-level label checking.
    Skips windows where the last day has no valid label (all-NaN band 7).
    This ensures every training sample has a verified ground truth.
    """
    def __init__(self, fire_dirs, time_steps, interval, patch_size,
                 means, stds, augment=False, min_fire_px=10, max_neg_ratio=2):
        self.T = time_steps
        self.ps = patch_size
        self.means = means
        self.stds = stds
        self.augment = augment
        self.samples = []
        self._build_index(fire_dirs, interval, min_fire_px, max_neg_ratio)

    def _build_index(self, fire_dirs, interval, min_fire_px, max_neg_ratio):
        n_pos = n_neg = n_neg_kept = skipped = no_label_days = 0
        rng = random.Random(cfg.SEED)

        for i, fd in enumerate(fire_dirs):
            day_files = sorted(glob.glob(os.path.join(fd, "VIIRS_Day", "*.tif")))
            if len(day_files) < self.T:
                skipped += 1; continue
            try:
                with rasterio.open(day_files[0]) as src:
                    if src.count < 7: skipped += 1; continue
                    H, W = src.height, src.width
            except Exception:
                skipped += 1; continue
            if H < self.ps or W < self.ps:
                skipped += 1; continue

            start = 0
            while start + self.T <= len(day_files):
                last_day = day_files[start + self.T - 1]

                # KEY FIX: check if last day has valid label
                if not check_day_has_label(last_day):
                    no_label_days += 1
                    start += interval
                    continue

                lbl = None
                try:
                    with rasterio.open(last_day) as src:
                        if src.count >= 7:
                            b7 = src.read(7).astype(np.float32)
                            if np.isnan(b7).sum() < b7.size:
                                lbl = (b7 >= 7).astype(np.float32)
                except Exception: pass

                if lbl is None:
                    no_label_days += 1
                    start += interval
                    continue

                r0 = (H - self.ps) // 2; c0 = (W - self.ps) // 2
                fire_px = int(lbl[r0:r0+self.ps, c0:c0+self.ps].sum())
                is_pos = fire_px >= min_fire_px

                if is_pos:
                    n_pos += 1; keep = True
                else:
                    n_neg += 1
                    keep = rng.random() < 1.0 / (max_neg_ratio + 1)
                    if keep: n_neg_kept += 1

                if keep:
                    self.samples.append({"fd": fd, "files": day_files,
                                         "start": start, "H": H, "W": W})
                start += interval

            if (i+1) % 20 == 0 or (i+1) == len(fire_dirs):
                print(f"\r  Index: {i+1}/{len(fire_dirs)} | {len(self.samples)} samp",
                      end="", flush=True)

        print(f"\n  Done: {len(self.samples)} samples "
              f"(pos={n_pos}, neg_kept={n_neg_kept}/{n_neg}, "
              f"skip={skipped}, days_no_label={no_label_days})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        fd, H, W = s["fd"], s["H"], s["W"]
        win = s["files"][s["start"]:s["start"] + self.T]

        frames, label = [], None
        for t, dp in enumerate(win):
            is_last = (t == len(win) - 1)
            if is_last:
                fr, label = load_frame(fd, dp, return_label=True)
            else:
                fr = load_frame(fd, dp)
            frames.append(fr[:, :H, :W])

        if label is None:
            label = np.zeros((H, W), dtype=np.float32)
        label = label[:H, :W]

        stack = np.stack(frames, axis=0)
        stack = (stack - self.means[None, :, None, None]) / \
                (self.stds[None, :, None, None] + 1e-8)
        stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0)

        # Center crop
        r0 = (H - self.ps) // 2; c0 = (W - self.ps) // 2
        stack = stack[:, :, r0:r0+self.ps, c0:c0+self.ps]
        label = label[r0:r0+self.ps, c0:c0+self.ps]

        # Augmentation
        if self.augment:
            if random.random() > 0.5:
                stack = np.flip(stack, axis=-1).copy()
                label = np.flip(label, axis=-1).copy()
            if random.random() > 0.5:
                stack = np.flip(stack, axis=-2).copy()
                label = np.flip(label, axis=-2).copy()
            k = random.randint(0, 3)
            if k:
                stack = np.rot90(stack, k, axes=(-2, -1)).copy()
                label = np.rot90(label, k, axes=(0, 1)).copy()

        x = torch.from_numpy(stack.transpose(1, 0, 2, 3).copy()).float()
        y = torch.from_numpy(label.copy()).long()
        return x, y


print("\nBuilding CLEAN train index...")
train_ds = AFDatasetClean(train_fires, cfg.TS_LENGTH, cfg.TRAIN_INTERVAL, cfg.IMAGE_SIZE,
                          cfg.MEAN, cfg.STD, augment=True,
                          min_fire_px=cfg.MIN_FIRE_PX, max_neg_ratio=cfg.MAX_NEG_RATIO)

print("\nBuilding CLEAN val index...")
val_ds = AFDatasetClean(val_fires, cfg.TS_LENGTH, cfg.TRAIN_INTERVAL, cfg.IMAGE_SIZE,
                        cfg.MEAN, cfg.STD, augment=False,
                        min_fire_px=cfg.MIN_FIRE_PX, max_neg_ratio=cfg.MAX_NEG_RATIO)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True,
                          drop_last=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True,
                        persistent_workers=True)

print(f"\nClean train: {len(train_ds)} samples, {len(train_loader)} bat/ep")
print(f"Clean val:   {len(val_ds)} samples, {len(val_loader)} bat/ep")

xb, yb = next(iter(train_loader))
print(f"x: {tuple(xb.shape)} | y: {tuple(yb.shape)} | "
      f"y unique: {yb.unique().tolist()} | fire%: {(yb==1).float().mean():.4f}")
print(f"\nCell 3 done in {time.time()-PIPELINE_START:.0f}s")


Original:  151 fires
Excluded:  19 fires (zero labels)
Clean train: 120 fires
Clean val:   12 fires
Removed from train: 18 fires
Removed from val:   1 fires

Building CLEAN train index...
  Index: 120/120 | 1719 samp
  Done: 1719 samples (pos=1602, neg_kept=117/315, skip=0, days_no_label=143)

Building CLEAN val index...
  Index: 12/12 | 203 samp
  Done: 203 samples (pos=190, neg_kept=13/35, skip=0, days_no_label=25)

Clean train: 1719 samples, 214 bat/ep
Clean val:   203 samples, 26 bat/ep
x: (8, 8, 2, 256, 256) | y: (8, 256, 256) | y unique: [0, 1] | fire%: 0.0304

Cell 3 done in 565s


## Cell 4: Model + Loss

Identical to v1. The architecture was never the problem -- the data was.


In [5]:
class SEBlock3D(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(nn.Linear(ch, ch//r, bias=False), nn.ReLU(True),
                                nn.Linear(ch//r, ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1, 1)


class ResBlock3D(nn.Module):
    def __init__(self, ic, oc, r=8, dr=0.1):
        super().__init__()
        self.c1 = nn.Conv3d(ic, oc, (1,3,3), padding=(0,1,1), bias=False)
        self.b1 = nn.BatchNorm3d(oc)
        self.c2 = nn.Conv3d(oc, oc, (1,3,3), padding=(0,1,1), bias=False)
        self.b2 = nn.BatchNorm3d(oc)
        self.se = SEBlock3D(oc, r)
        self.relu = nn.ReLU(True)
        self.drop = nn.Dropout3d(dr) if dr > 0 else nn.Identity()
        self.skip = (nn.Sequential(nn.Conv3d(ic, oc, 1, bias=False),
                     nn.BatchNorm3d(oc)) if ic != oc else nn.Identity())

    def forward(self, x):
        r = self.skip(x)
        o = self.relu(self.b1(self.c1(x)))
        o = self.drop(o)
        o = self.b2(self.c2(o))
        o = self.se(o)
        return self.relu(o + r)


class SEUNet3D(nn.Module):
    def __init__(self, ic=8, nc=1, ec=(64,128,256,512), r=8, dr=0.1):
        super().__init__()
        self.e1 = ResBlock3D(ic, ec[0], r, dr)
        self.e2 = ResBlock3D(ec[0], ec[1], r, dr)
        self.e3 = ResBlock3D(ec[1], ec[2], r, dr)
        self.e4 = ResBlock3D(ec[2], ec[3], r, dr)
        self.pool = nn.MaxPool3d((1,2,2), stride=(1,2,2))
        self.bot = ResBlock3D(ec[3], ec[3]*2, r, dr)

        self.u4 = nn.ConvTranspose3d(ec[3]*2, ec[3], (1,2,2), stride=(1,2,2))
        self.d4 = ResBlock3D(ec[3]*2, ec[3], r, dr)
        self.u3 = nn.ConvTranspose3d(ec[3], ec[2], (1,2,2), stride=(1,2,2))
        self.d3 = ResBlock3D(ec[2]*2, ec[2], r, dr)
        self.u2 = nn.ConvTranspose3d(ec[2], ec[1], (1,2,2), stride=(1,2,2))
        self.d2 = ResBlock3D(ec[1]*2, ec[1], r, dr)
        self.u1 = nn.ConvTranspose3d(ec[1], ec[0], (1,2,2), stride=(1,2,2))
        self.d1 = ResBlock3D(ec[0]*2, ec[0], r, dr)

        self.final = nn.Conv3d(ec[0], nc, 1)
        self.ds3 = nn.Conv3d(ec[2], nc, 1)  # deep supervision

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.bot(self.pool(e4))

        d4 = self.d4(torch.cat([self.u4(b), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))

        out = self.final(d1)
        if self.training:
            ds = F.interpolate(self.ds3(d3), size=out.shape[2:],
                               mode="trilinear", align_corners=False)
            return out, ds
        return out


class DiceFocalLoss(nn.Module):
    def __init__(self, dw=0.5, fw=0.5, gamma=2.0, alpha=0.75, dsw=0.3):
        super().__init__()
        self.dw, self.fw, self.gamma, self.alpha, self.dsw = dw, fw, gamma, alpha, dsw

    def _dice(self, p, t):
        ps = torch.sigmoid(p).reshape(-1); tf = t.reshape(-1)
        return 1 - (2*(ps*tf).sum()+1) / (ps.sum()+tf.sum()+1)

    def _focal(self, p, t):
        bce = F.binary_cross_entropy_with_logits(p, t, reduction="none")
        pt = torch.sigmoid(p)*t + (1-torch.sigmoid(p))*(1-t)
        at = self.alpha*t + (1-self.alpha)*(1-t)
        return (at * (1-pt)**self.gamma * bce).mean()

    def _loss(self, p, t):
        return self.dw*self._dice(p, t) + self.fw*self._focal(p, t)

    def forward(self, preds, target):
        main = preds[0] if isinstance(preds, tuple) else preds
        ds = preds[1] if isinstance(preds, tuple) else None
        pred_last = main[:, :, -1, :, :]
        tgt = target.unsqueeze(1).float()
        loss = self._loss(pred_last, tgt)
        if ds is not None:
            loss += self.dsw * self._loss(ds[:, :, -1, :, :], tgt)
        return loss


model = SEUNet3D(ic=cfg.N_CHANNELS, nc=1, ec=tuple(cfg.ENCODER_CHANNELS),
                 r=cfg.SE_REDUCTION, dr=cfg.DROPOUT).to(DEVICE)
criterion = DiceFocalLoss(cfg.DICE_WEIGHT, cfg.FOCAL_WEIGHT,
                          cfg.FOCAL_GAMMA, cfg.FOCAL_ALPHA, cfg.DS_WEIGHT)
n_params = sum(p.numel() for p in model.parameters())

print(f"Model: SE-UNet3D v6 (identical backbone to v1)")
print(f"Params: {n_params:,} ({n_params/1e6:.2f}M)")
print(f"Difference from v1: CLEAN DATA ONLY")
print(f"\nCell 4 done in {time.time()-PIPELINE_START:.0f}s")


Model: SE-UNet3D v6 (identical backbone to v1)
Params: 32,876,034 (32.88M)
Difference from v1: CLEAN DATA ONLY

Cell 4 done in 567s


## Cell 5: Training

Same recipe as v1: OneCycleLR, AMP, grad clip.
Expect faster convergence and higher F1 since every sample
now has verified labels.


In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE,
                               weight_decay=cfg.WEIGHT_DECAY)
scheduler = OneCycleLR(optimizer, max_lr=cfg.LEARNING_RATE,
                       steps_per_epoch=len(train_loader),
                       epochs=cfg.MAX_EPOCHS, pct_start=0.1, anneal_strategy="cos")
scaler = GradScaler(enabled=cfg.USE_AMP)

history = {"train_loss":[], "val_loss":[], "val_f1":[], "val_iou":[],
           "val_prec":[], "val_rec":[], "lr":[], "epoch_time":[]}

best_f1 = best_iou = 0.0
best_epoch = 0
patience_ctr = 0
THRESHOLD = 0.5
train_start = time.time()

print(f"Training on CLEAN data: {len(train_loader)} bat/ep x {cfg.MAX_EPOCHS} ep")
print(f"{'Ep':>3} {'TrL':>7} {'VaL':>7} {'F1':>7} {'IoU':>7} "
      f"{'P':>6} {'R':>6} {'LR':>9} {'T':>4}")
print("=" * 72)

for epoch in range(cfg.MAX_EPOCHS):
    ep_start = time.time()
    
    # Time limit safeguard
    if (time.time()-PIPELINE_START)/3600 > 10.5:
        print(f"\nTime limit. Stopping.")
        break

    model.train()
    rloss = 0.0
    for xb, yb in tqdm(train_loader, desc=f"E{epoch+1:2d} Tr", leave=False, ncols=80):
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with autocast(enabled=cfg.USE_AMP):
            out = model(xb)
            loss = criterion(out, yb)
            
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        rloss += loss.item()
        
    tl = rloss / len(train_loader)

    model.eval()
    vloss = 0.0
    ap, al = [], []
    with torch.no_grad():
        for xb, yb in tqdm(val_loader, desc=f"E{epoch+1:2d} Va", leave=False, ncols=80):
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            with autocast(enabled=cfg.USE_AMP):
                out = model(xb)
                loss = criterion(out, yb)
                
            vloss += loss.item()
            logits = out[0] if isinstance(out, tuple) else out
            p = (torch.sigmoid(logits[:, 0, -1]) > THRESHOLD).cpu().numpy().flatten()
            ap.append(p)
            al.append(yb.cpu().numpy().flatten())

    vl = vloss / max(len(val_loader), 1)
    ap, al = np.concatenate(ap), np.concatenate(al)
    
    vf1 = f1_score(al, ap, zero_division=0.0)
    viou = jaccard_score(al, ap, zero_division=0.0)
    vp = precision_score(al, ap, zero_division=0.0)
    vr = recall_score(al, ap, zero_division=0.0)
    lr_now = optimizer.param_groups[0]["lr"]
    etime = time.time() - ep_start

    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["val_f1"].append(vf1)
    history["val_iou"].append(viou)
    history["val_prec"].append(vp)
    history["val_rec"].append(vr)
    history["lr"].append(lr_now)
    history["epoch_time"].append(etime)

    note = ""
    if vf1 > best_f1:
        best_f1, best_iou, best_epoch = vf1, viou, epoch+1
        patience_ctr = 0
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                     "f1": vf1, "iou": viou},
                    os.path.join(cfg.SAVE_DIR, "best_v6.pt"))
        note = " << BEST"
    else:
        patience_ctr += 1

    elapsed_m = (time.time()-PIPELINE_START)/60
    print(f"{epoch+1:3d} {tl:7.4f} {vl:7.4f} {vf1:7.4f} {viou:7.4f} "
          f"{vp:6.3f} {vr:6.3f} {lr_now:9.1e} {etime:4.0f}s "
          f"[{elapsed_m:.0f}m]{note}")

    # Notifies if patience is exceeded, but intentionally does NOT break the loop
    if hasattr(cfg, 'PATIENCE') and patience_ctr >= cfg.PATIENCE: 
        print(f'  [!] No improvement for {patience_ctr} epochs (continuing training...)')

# --- END OF FOR LOOP ---

# Save the final model state after all epochs are complete
torch.save({"epoch": epoch, "model_state_dict": model.state_dict()},
           os.path.join(cfg.SAVE_DIR, "last_v6.pt"))

total_train = time.time() - train_start
print(f"\n{'='*72}")
print(f"Training: {total_train/3600:.2f}h ({len(history['train_loss'])} ep)")
print(f"Best F1:  {best_f1:.4f} (ep {best_epoch}) | IoU: {best_iou:.4f}")
print(f"Paper: 0.823 | v1 (dirty): 0.818 | v4 (dirty): 0.817")
print(f"Delta vs paper: {best_f1-0.823:+.4f}")
if best_f1 > 0.823: 
    print(">>> BEAT THE PAPER <<<")
print(f"{'='*72}")

Training on CLEAN data: 214 bat/ep x 80 ep
 Ep     TrL     VaL      F1     IoU      P      R        LR    T


E 1 Tr:   0%|                                           | 0/214 [00:00<?, ?it/s]

E 1 Va:   0%|                                            | 0/26 [00:00<?, ?it/s]

  1  0.6408  0.4974  0.4151  0.2619  0.271  0.891   3.8e-05  314s [15m] << BEST


E 2 Tr:   0%|                                           | 0/214 [00:00<?, ?it/s]

...

E79 Va:   0%|                                            | 0/26 [00:00<?, ?it/s]

 79  0.1216  0.1253  0.8138  0.6861  0.844  0.786   2.4e-07  297s [395m]


E80 Tr:   0%|                                           | 0/214 [00:00<?, ?it/s]

E80 Va:   0%|                                            | 0/26 [00:00<?, ?it/s]

 80  0.1216  0.1279  0.7989  0.6651  0.863  0.744   2.0e-09  295s [400m]
  [!] No improvement for 20 epochs (continuing training...)

Training: 6.50h (80 ep)
Best F1:  0.8240 (ep 60) | IoU: 0.7006
Paper: 0.823 | v1 (dirty): 0.818 | v4 (dirty): 0.817
Delta vs paper: +0.0010
>>> BEAT THE PAPER <<<


## Cell 6: Threshold Sweep


In [7]:
print("Loading best model for threshold sweep...")
ckpt = torch.load(os.path.join(cfg.SAVE_DIR, "best_v6.pt"),
                   map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded ep {ckpt['epoch']+1}, F1={ckpt['f1']:.4f}")

model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for xb, yb in tqdm(val_loader, desc="Preds", ncols=80):
        xb = xb.to(DEVICE, non_blocking=True)
        with autocast(enabled=cfg.USE_AMP):
            out = model(xb)
        logits = out[0] if isinstance(out, tuple) else out
        all_probs.append(torch.sigmoid(logits[:, 0, -1]).cpu().numpy().flatten())
        all_labels.append(yb.cpu().numpy().flatten())
all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

thresholds = np.arange(0.25, 0.71, 0.02)
sweep = []
for thr in thresholds:
    p = (all_probs > thr).astype(float)
    sweep.append({"thr": thr,
                  "f1": f1_score(all_labels, p, zero_division=0.0),
                  "iou": jaccard_score(all_labels, p, zero_division=0.0),
                  "prec": precision_score(all_labels, p, zero_division=0.0),
                  "rec": recall_score(all_labels, p, zero_division=0.0)})

sweep_df = pd.DataFrame(sweep)
best_row = sweep_df.loc[sweep_df["f1"].idxmax()]
opt_thr = float(best_row["thr"])
opt_f1 = float(best_row["f1"])
opt_iou = float(best_row["iou"])

print(f"\n{'Thr':>5} {'F1':>7} {'IoU':>7} {'P':>6} {'R':>6}")
print("-" * 38)
for _, r in sweep_df.iterrows():
    m = " <<<" if r["thr"] == opt_thr else ""
    print(f"{r['thr']:5.2f} {r['f1']:7.4f} {r['iou']:7.4f} "
          f"{r['prec']:6.3f} {r['rec']:6.3f}{m}")

print(f"\nOptimal: thr={opt_thr:.2f}, F1={opt_f1:.4f}")
print(f"Default 0.50: F1={ckpt['f1']:.4f}, Gain: {opt_f1-ckpt['f1']:+.4f}")
if opt_f1 > best_f1:
    best_f1 = opt_f1; best_iou = opt_iou

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep_df["thr"], sweep_df["f1"], "g-o", lw=2, ms=4, label="F1")
ax.plot(sweep_df["thr"], sweep_df["iou"], "m-s", lw=1.5, ms=3, label="IoU")
ax.plot(sweep_df["thr"], sweep_df["prec"], "c--", lw=1, label="Prec")
ax.plot(sweep_df["thr"], sweep_df["rec"], "y--", lw=1, label="Rec")
ax.axvline(opt_thr, color="red", ls=":", label=f"Opt ({opt_thr:.2f})")
ax.axhline(0.823, color="gray", ls="--", alpha=0.5, label="Paper")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
ax.set_title("Threshold Sweep (Clean Val)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, "plots", "threshold_v6.png"), dpi=150)
plt.show(); plt.close()
print("Saved: plots/threshold_v6.png")


Loading best model for threshold sweep...
Loaded ep 60, F1=0.8240


Preds:   0%|                                             | 0/26 [00:00<?, ?it/s]


  Thr      F1     IoU      P      R
--------------------------------------
 0.25  0.8248  0.7018  0.814  0.836 <<<
 0.27  0.8246  0.7016  0.815  0.835
 0.29  0.8247  0.7017  0.816  0.834
 0.31  0.8246  0.7015  0.817  0.833
 0.33  0.8246  0.7016  0.818  0.831
 0.35  0.8246  0.7016  0.819  0.830
 0.37  0.8244  0.7013  0.820  0.829
 0.39  0.8244  0.7012  0.821  0.828
 0.41  0.8244  0.7012  0.822  0.827
 0.43  0.8244  0.7012  0.822  0.826
 0.45  0.8242  0.7009  0.823  0.825
 0.47  0.8241  0.7008  0.824  0.824
 0.49  0.8240  0.7006  0.825  0.823
 0.51  0.8239  0.7006  0.826  0.822
 0.53  0.8238  0.7004  0.827  0.821
 0.55  0.8237  0.7002  0.827  0.820
 0.57  0.8235  0.6999  0.828  0.819
 0.59  0.8235  0.6999  0.829  0.818
 0.61  0.8234  0.6998  0.830  0.817
 0.63  0.8232  0.6995  0.830  0.816
 0.65  0.8230  0.6992  0.831  0.815
 0.67  0.8229  0.6991  0.832  0.814
 0.69  0.8229  0.6990  0.833  0.813

Optimal: thr=0.25, F1=0.8248
Default 0.50: F1=0.8240, Gain: +0.0008
Saved: plots/threshold_

## Cell 7: Plots + Comparison


In [8]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("SE-UNet3D v6 -- Clean Data Training", fontsize=14, fontweight="bold")
ep = range(1, len(history["train_loss"]) + 1)
axes[0,0].plot(ep, history["train_loss"], "b-", label="Train")
axes[0,0].plot(ep, history["val_loss"], "r-", label="Val")
axes[0,0].set_title("Loss"); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)
axes[0,1].plot(ep, history["val_f1"], "g-", lw=2)
axes[0,1].axhline(0.823, color="red", ls="--", label="Paper (0.823)")
axes[0,1].axhline(0.818, color="orange", ls=":", label="v1 dirty (0.818)")
axes[0,1].set_title("Val F1"); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)
axes[0,2].plot(ep, history["val_iou"], "m-", lw=2)
axes[0,2].axhline(0.727, color="red", ls="--", label="Paper")
axes[0,2].set_title("Val IoU"); axes[0,2].legend(); axes[0,2].grid(alpha=0.3)
axes[1,0].plot(ep, history["val_prec"], "c-", label="P")
axes[1,0].plot(ep, history["val_rec"], "y-", label="R")
axes[1,0].set_title("P & R"); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)
axes[1,1].plot(ep, history["lr"], "k-")
axes[1,1].set_title("LR"); axes[1,1].set_yscale("log"); axes[1,1].grid(alpha=0.3)
sc = axes[1,2].scatter(history["val_iou"], history["val_f1"], c=list(ep), cmap="viridis", s=20)
axes[1,2].axhline(0.823, color="red", ls="--", alpha=0.5)
axes[1,2].axvline(0.727, color="red", ls="--", alpha=0.5)
axes[1,2].set_title("F1 vs IoU"); plt.colorbar(sc, ax=axes[1,2], label="Epoch")
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, "plots", "curves_v6.png"), dpi=150, bbox_inches="tight")
plt.show(); plt.close()

models_cmp = OrderedDict([
    ("U-Net (2D)",            (0.731, 0.605)),
    ("Att-UNet (2D)",         (0.763, 0.648)),
    ("UNETR-2D",              (0.733, 0.621)),
    ("SwinUNETR-2D",          (0.774, 0.660)),
    ("GRU-3",                 (0.713, 0.601)),
    ("LSTM-3",                (0.765, 0.654)),
    ("T4Fire",                (0.802, 0.700)),
    ("U-Net-3D",              (0.748, 0.628)),
    ("Att-UNet-3D",           (0.770, 0.654)),
    ("UNETR-3D",              (0.811, 0.706)),
    ("SwinUNETR-3D (TS=6)",   (0.797, 0.688)),
    ("SwinUNETR-3D (TS=2)",   (0.823, 0.727)),
    ("Ours v1 (noisy data)",  (0.818, 0.692)),
    ("Ours v6 (clean data)",  (float(best_f1), float(best_iou))),
])
names = list(models_cmp.keys()); f1s = [v[0] for v in models_cmp.values()]
colors = ["#6baed6"]*(len(names)-2) + ["#fdae6b", "#e6550d"]
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(names, f1s, color=colors, edgecolor="gray", alpha=0.85)
ax.set_xlabel("F1"); ax.set_title("F1 Comparison", fontweight="bold")
ax.grid(axis="x", alpha=0.3)
for i, v in enumerate(f1s):
    ax.text(v+0.003, i, f"{v:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, "plots", "comparison_v6.png"), dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print("All plots saved.")


All plots saved.


## Cell 8: Save Results


In [9]:
pd.DataFrame(history).to_csv(os.path.join(cfg.OUTPUT_DIR, "history_v6.csv"), index_label="epoch")

results = {
    "model": "SE-UNet3D-v6-clean",
    "n_params": int(n_params), "n_params_M": round(n_params/1e6, 2),
    "ts_length": cfg.TS_LENGTH, "patch_size": cfg.IMAGE_SIZE,
    "batch_size": cfg.BATCH_SIZE,
    "epochs_run": len(history["train_loss"]),
    "best_epoch": int(best_epoch),
    "best_f1": round(float(best_f1), 4),
    "best_iou": round(float(best_iou), 4),
    "optimal_threshold": round(float(opt_thr), 2),
    "paper_f1": 0.823, "paper_iou": 0.727,
    "v1_f1_dirty": 0.818, "v1_iou_dirty": 0.692,
    "delta_vs_paper": round(float(best_f1) - 0.823, 4),
    "delta_vs_v1": round(float(best_f1) - 0.818, 4),
    "beat_paper": bool(float(best_f1) > 0.823),
    "train_fires_total": len(numeric_ids) - len(cfg.VAL_IDS),
    "train_fires_clean": len(train_fires),
    "train_fires_excluded": len(cfg.NO_LABEL_IDS) - 1,  # minus the 1 val fire
    "val_fires_total": len(cfg.VAL_IDS),
    "val_fires_clean": len(val_fires),
    "train_time_h": round(total_train/3600, 2),
    "key_change": "Excluded 19 fires with missing AF labels (band 7 all-NaN)",
}
with open(os.path.join(cfg.OUTPUT_DIR, "results_v6.json"), "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

total = time.time() - PIPELINE_START
print(f"\n{'='*72}")
print(f"v6 FINAL: F1={float(best_f1):.4f} IoU={float(best_iou):.4f}")
print(f"Paper: 0.823/0.727 | v1 dirty: 0.818/0.692")
print(f"Clean train: {len(train_fires)} fires | Clean val: {len(val_fires)} fires")
print(f"Threshold: {float(opt_thr):.2f} | Time: {total/3600:.2f}h")
if float(best_f1) > 0.823:
    print(">>> BEAT THE PAPER <<<")
print(f"{'='*72}")


{
  "model": "SE-UNet3D-v6-clean",
  "n_params": 32876034,
  "n_params_M": 32.88,
  "ts_length": 2,
  "patch_size": 256,
  "batch_size": 8,
  "epochs_run": 80,
  "best_epoch": 60,
  "best_f1": 0.8248,
  "best_iou": 0.7018,
  "optimal_threshold": 0.25,
  "paper_f1": 0.823,
  "paper_iou": 0.727,
  "v1_f1_dirty": 0.818,
  "v1_iou_dirty": 0.692,
  "delta_vs_paper": 0.0018,
  "delta_vs_v1": 0.0068,
  "beat_paper": true,
  "train_fires_total": 138,
  "train_fires_clean": 120,
  "train_fires_excluded": 18,
  "val_fires_total": 13,
  "val_fires_clean": 12,
  "train_time_h": 6.5,
  "key_change": "Excluded 19 fires with missing AF labels (band 7 all-NaN)"
}

v6 FINAL: F1=0.8248 IoU=0.7018
Paper: 0.823/0.727 | v1 dirty: 0.818/0.692
Clean train: 120 fires | Clean val: 12 fires
Threshold: 0.25 | Time: 6.70h
>>> BEAT THE PAPER <<<
